In [ ]:
#質量保存を確認するコード（jeans_3d-test25でmassが吹っ飛んだとき、質量保存しているかどうか確認するために使ったコード）

import numpy as np
import matplotlib.pyplot as plt
import os

# ========= 入力ファイル =========
hst_file = os.path.expanduser(
    "~/athenapp/results/〇〇/Jeans.hst"
)

# ========= 出力ディレクトリ =========
output_dir = "./total_mass"
os.makedirs(output_dir, exist_ok=True)

# ========= データ読み込み =========
# コメント行 (#) を無視
data = np.loadtxt(hst_file, comments="#")

time = data[:, 0]
mass = data[:, 2]

# ========= mass 消失チェック =========
# 閾値（「ほぼ0」とみなす基準）
mass_threshold = 1e-10

bad = (
    (mass < mass_threshold) |
    (~np.isfinite(mass))
)

if np.any(bad):
    print("⚠️ mass が異常になったタイムステップ:")
    for t, m in zip(time[bad], mass[bad]):
        print(f"  t = {t:.6e}, mass = {m}")
else:
    print("✅ mass は全ステップで正常")

# ========= プロット =========
plt.figure(figsize=(6, 4))

plt.plot(
    time,
    mass,
    marker="o",
    linestyle="-",
    markersize=3,   # ← 今までが 9 くらいなら 1/3
)

plt.xlabel("Time")
plt.ylabel("Total Mass")
plt.yscale("log")
plt.title("Total Mass vs File_steps")
plt.grid(True, which="both", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(
    os.path.join(output_dir, "mass_vs_filesteps.png"),
    dpi=200
)
plt.show()

In [ ]:
#各タイムステップで計算領域の質量保存（dM/dt+flux=0）を確認するコード

import pyvista as pv
import numpy as np
import glob
import re
import os

# =====================================================
# 設定
# =====================================================

vtk_dir = os.path.expanduser("~/athenapp/results/〇〇")

xmin,xmax = 0.0,8.0　　#計算領域に合わせる
ymin,ymax = 0.0,8.0
zmin,zmax = 0.0,8.0

# =====================================================
# VTK ファイル取得
# =====================================================

vtk_files = sorted(glob.glob(os.path.join(vtk_dir,"Jeans.block*.out2.*.vtk")))

print("Total VTK files:",len(vtk_files))

# timestep 抽出
steps = sorted(set(re.search(r"out2\.(\d+)",f).group(1) for f in vtk_files))

print("Detected timesteps:",len(steps))

# =====================================================
# 保存用配列
# =====================================================

time_list = []
mass_list = []
flux_list = []

# =====================================================
# timestep ループ
# =====================================================

for step in steps:

    files = sorted(glob.glob(os.path.join(vtk_dir,f"Jeans.block*.out2.{step}.vtk")))

    total_mass = 0.0
    total_flux = 0.0

    for f in files:

        grid = pv.read(f)

        rho = grid.cell_data["dens"]
        mom = grid.cell_data["mom"]

        vx = mom[:,0] / rho
        vy = mom[:,1] / rho
        vz = mom[:,2] / rho

        centers = grid.cell_centers().points

        x = centers[:,0]
        y = centers[:,1]
        z = centers[:,2]

        bounds = grid.bounds

        dx = (bounds[1]-bounds[0]) / grid.dimensions[0]
        dy = (bounds[3]-bounds[2]) / grid.dimensions[1]
        dz = (bounds[5]-bounds[4]) / grid.dimensions[2]

        cell_vol = dx*dy*dz

        # ------------------------------------------------
        # total mass
        # ------------------------------------------------

        total_mass += np.sum(rho * cell_vol)

        # ------------------------------------------------
        # flux
        # ------------------------------------------------

        mask = np.isclose(x,xmin)
        total_flux += np.sum(rho[mask]*vx[mask]*dy*dz)

        mask = np.isclose(x,xmax)
        total_flux += np.sum(rho[mask]*vx[mask]*dy*dz)

        mask = np.isclose(y,ymin)
        total_flux += np.sum(rho[mask]*vy[mask]*dx*dz)

        mask = np.isclose(y,ymax)
        total_flux += np.sum(rho[mask]*vy[mask]*dx*dz)

        mask = np.isclose(z,zmin)
        total_flux += np.sum(rho[mask]*vz[mask]*dx*dy)

        mask = np.isclose(z,zmax)
        total_flux += np.sum(rho[mask]*vz[mask]*dx*dy)

    time_list.append(int(step))
    mass_list.append(total_mass)
    flux_list.append(total_flux)

# =====================================================
# dM/dt 計算
# =====================================================

mass_list = np.array(mass_list)
flux_list = np.array(flux_list)
time_list = np.array(time_list)

dMdt = np.zeros_like(mass_list)

for i in range(1,len(mass_list)):

    dMdt[i] = mass_list[i] - mass_list[i-1]

# =====================================================
# 結果表示
# =====================================================

print("\n=== Mass evolution ===")

for i in range(len(time_list)):

    print(f"step {time_list[i]:5d}  M={mass_list[i]:.6e}  flux={flux_list[i]:.6e}  dM={dMdt[i]:.6e}")

# =====================================================
# 保存則チェック
# =====================================================

print("\n=== Conservation check ===")

error = dMdt + flux_list

print("mean error:",np.mean(np.abs(error)))
print("max  error:",np.max(np.abs(error)))